# Notebook 02 — Feature Engineering and Modeling
**Level:** 2 (mandatory)  
**Depends on:** `01_data_understanding.ipynb` (must have run preprocessing pipeline)  

---

## Objectives
1. Load the cleaned dataset.
2. Engineer temporal and operational features.
3. Create a strict temporal split (train / val / test).
4. Train multiple models.
5. Evaluate and compare on validation set.
6. Select and persist the best model.
7. Run final evaluation on the held-out test set.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import warnings
warnings.filterwarnings('ignore')

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns

from src.config import CFG
from src.feature_engineering import (
    build_features,
    temporal_split,
    prepare_Xy,
    run_feature_engineering_pipeline,
)
from src.train import train_all_models, save_model, log_to_mlflow
from src.evaluate import (
    compute_metrics,
    residuals_dataframe,
    build_metrics_table,
    save_metrics_table,
    error_by_time_block,
)

sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 120
FIGURES_PATH = Path(CFG['paths']['reports_figures'])
TARGET = CFG['target']
print(f'Target: {TARGET}')

---
## 1. Load Cleaned Data

In [ ]:
interim_path = Path(CFG['paths']['data_interim']) / 'data_cleaned.parquet'

if not interim_path.exists():
    raise FileNotFoundError(
        'Cleaned data not found. Run notebook 01 first to generate data_cleaned.parquet.'
    )

df_clean = pd.read_parquet(interim_path)
print(f'Loaded cleaned data: {df_clean.shape}')
print(f'Date range: {df_clean.index.min()} → {df_clean.index.max()}')

---
## 2. Feature Engineering

### Leakage Prevention Strategy

> - **Lab measurements** (`% Iron Concentrate`, `% Silica Concentrate`) are excluded
>   from the feature set because they are not available at inference time.
> - **Lag features** use `shift(n)` with `n ≥ 1` — the current row's value is never
>   used to predict itself.
> - **Rolling statistics** are computed on `shift(1)` of each variable, so the rolling
>   window never includes the current observation.
> - **Scalers** (if used) are fitted exclusively on the training split.

### Lag Design Rationale
The dataset has a 20-minute sampling interval. The lag periods `[1, 3, 6, 12]` correspond
to approximately: 20 min, 1 h, 2 h, and 4 h lookbacks — covering the typical process
response time for reagent and operational changes to propagate to concentrate quality.

In [ ]:
# Run the full feature engineering + split + save pipeline
X_train, X_val, X_test, y_train, y_val, y_test = run_feature_engineering_pipeline(
    df_clean, cfg=CFG, save=True
)

print(f'\nX_train shape: {X_train.shape}')
print(f'X_val   shape: {X_val.shape}')
print(f'X_test  shape: {X_test.shape}')
print(f'\nFeature count: {X_train.shape[1]}')
print(f'\nSample features: {list(X_train.columns[:8])}...')

---
## 3. Baseline Reference

We load the baseline metrics saved in notebook 01 as our performance floor.

In [ ]:
metrics_path = Path(CFG['paths']['reports_metrics'])
with open(metrics_path / 'baseline_metrics.json', 'r') as f:
    baseline_metrics = json.load(f)

print('Baseline metrics (predict training mean):')
for split_name, m in baseline_metrics.items():
    print(f'  {split_name}: {m}')

---
## 4. Model Training

In [ ]:
# Train all models — validation set is used for EVALUATION only, never for fitting
fitted_models, all_metrics = train_all_models(
    X_train, y_train, X_val, y_val, cfg=CFG
)

In [ ]:
# Build and display validation metrics table
val_results = {name: metrics['val'] for name, metrics in all_metrics.items()}
metrics_table = build_metrics_table(val_results)
print('\n=== Validation Metrics ===')
metrics_table

---
## 5. Model Evaluation Beyond a Single Metric

We analyse: real vs predicted, residuals, temporal error patterns, and block comparisons.

In [ ]:
# ── 5a. Real vs Predicted — all models on validation set ─────────────────
n_models = len(fitted_models)
fig, axes = plt.subplots(1, n_models, figsize=(5 * n_models, 4))
if n_models == 1:
    axes = [axes]

for ax, (name, model) in zip(axes, fitted_models.items()):
    y_pred = model.predict(X_val)
    ax.scatter(y_val, y_pred, alpha=0.3, s=8, color='steelblue')
    lims = [min(y_val.min(), y_pred.min()), max(y_val.max(), y_pred.max())]
    ax.plot(lims, lims, 'r--', lw=1, label='Perfect')
    r2 = compute_metrics(y_val, y_pred)['R2']
    ax.set_title(f'{name}\nR²={r2}', fontsize=10)
    ax.set_xlabel('Actual')
    ax.set_ylabel('Predicted')

plt.suptitle('Real vs Predicted — Validation Set', y=1.01, fontsize=12)
plt.tight_layout()
plt.savefig(FIGURES_PATH / 'real_vs_predicted_val.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── 5b. Residuals distribution ─────────────────────────────────────────────
fig, axes = plt.subplots(1, n_models, figsize=(5 * n_models, 4))
if n_models == 1:
    axes = [axes]

for ax, (name, model) in zip(axes, fitted_models.items()):
    y_pred = model.predict(X_val)
    res = y_val.values - y_pred
    ax.hist(res, bins=40, color='coral', edgecolor='white', alpha=0.8)
    ax.axvline(0, color='navy', lw=1.5, linestyle='--')
    ax.set_title(f'{name} — Residuals', fontsize=10)
    ax.set_xlabel('Residual (actual − predicted)')
    ax.set_ylabel('Count')

plt.suptitle('Residual Distributions — Validation Set', y=1.01, fontsize=12)
plt.tight_layout()
plt.savefig(FIGURES_PATH / 'residuals_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── 5c. Temporal residual error — select the best model ──────────────────
# Identify best model by validation RMSE
best_name = metrics_table.iloc[0]['Model']
best_model = fitted_models[best_name]
print(f'Best model by RMSE (val): {best_name}')

y_pred_best = best_model.predict(X_val)
res_df = residuals_dataframe(y_val, y_pred_best, split_name='val')

fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(res_df.index, res_df['residual'], lw=0.5, color='coral', alpha=0.7)
ax.axhline(0, color='navy', lw=1.2, linestyle='--')
ax.set_title(f'Temporal Residuals — {best_name} (Validation)', fontsize=12)
ax.set_xlabel('Date')
ax.set_ylabel('Residual')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
plt.tight_layout()
plt.savefig(FIGURES_PATH / 'temporal_residuals.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── 5d. Error by monthly block ────────────────────────────────────────────
monthly_error = error_by_time_block(res_df, freq='ME')

fig, ax = plt.subplots(figsize=(10, 4))
monthly_error['MAE'].plot(kind='bar', ax=ax, color='steelblue', edgecolor='white')
ax.set_title(f'MAE by Month — {best_name} (Validation)', fontsize=12)
ax.set_ylabel('MAE')
ax.set_xlabel('Month')
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig(FIGURES_PATH / 'mae_by_month.png', dpi=150, bbox_inches='tight')
plt.show()
monthly_error

---
## 6. Model Selection and Final Test Evaluation

**Model selection justification:**  
We select the model with the lowest RMSE on the **validation set** (unseen during training).
RMSE is prioritised because large quality deviations in the mining process carry
disproportionate operational costs. Final test set performance confirms generalization.

> The test set is evaluated **only once** here — after model selection is finalised —
> to avoid inadvertent optimisation toward the test set.

In [ ]:
# Final evaluation on test set (evaluated ONCE)
y_pred_test = best_model.predict(X_test)
test_metrics = compute_metrics(y_test, y_pred_test)

print(f'\n=== Final Test Set Metrics — {best_name} ===')
print(f'MAE:  {test_metrics["MAE"]}')
print(f'RMSE: {test_metrics["RMSE"]}')
print(f'R²:   {test_metrics["R2"]}')

print(f'\nBaseline Test (from nb01):')
print(f'  MAE:  {baseline_metrics["Test"]["MAE"]}')
print(f'  RMSE: {baseline_metrics["Test"]["RMSE"]}')
print(f'  R²:   {baseline_metrics["Test"]["R2"]}')

In [ ]:
# Add test metrics to the comparison table and save
test_results = {name: compute_metrics(y_test, model.predict(X_test))
                for name, model in fitted_models.items()}
full_table = build_metrics_table(test_results)
full_table.insert(0, 'split', 'test')

save_metrics_table(full_table, CFG)
full_table

In [ ]:
# Real vs Predicted — Test set (best model)
res_df_test = residuals_dataframe(y_test, y_pred_test, split_name='test')

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Scatter
axes[0].scatter(y_test, y_pred_test, alpha=0.3, s=8, color='steelblue')
lims = [min(y_test.min(), y_pred_test.min()), max(y_test.max(), y_pred_test.max())]
axes[0].plot(lims, lims, 'r--', lw=1)
axes[0].set_title(f'{best_name} — Real vs Predicted (Test)', fontsize=11)
axes[0].set_xlabel('Actual % Silica')
axes[0].set_ylabel('Predicted % Silica')

# Temporal prediction trace
axes[1].plot(res_df_test.index, res_df_test['y_true'], lw=0.5, label='Actual', color='navy')
axes[1].plot(res_df_test.index, res_df_test['y_pred'], lw=0.5, label='Predicted', color='coral', alpha=0.8)
axes[1].set_title(f'Temporal Forecast — {best_name} (Test)', fontsize=11)
axes[1].set_xlabel('Date')
axes[1].set_ylabel('% Silica Concentrate')
axes[1].legend()
axes[1].xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))

plt.tight_layout()
plt.savefig(FIGURES_PATH / 'real_vs_predicted_test.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 7. Save Selected Model

In [ ]:
model_path = save_model(best_model, model_name=best_name, folder='selected', cfg=CFG)

# Also save feature list alongside model for reproducibility
import json
feature_list_path = Path(CFG['paths']['models_selected']) / 'feature_columns.json'
with open(feature_list_path, 'w') as f:
    json.dump(list(X_train.columns), f, indent=2)
print(f'Feature list saved to: {feature_list_path}')

In [ ]:
# Log to MLflow
train_metrics_best = all_metrics[best_name]['train']
val_metrics_best   = all_metrics[best_name]['val']
log_to_mlflow(best_model, best_name, train_metrics_best, val_metrics_best, cfg=CFG)

---
## Summary — Level 2 Checklist

| Item | Status |
|---|---|
| Temporal split (train/val/test) | ✅ |
| No random split (`train_test_split` not used) | ✅ |
| Temporal features (hour, day, month, dayofweek, is_weekend) | ✅ |
| Lag features (operational variables, lag 1/3/6/12) | ✅ |
| Rolling mean and std features | ✅ |
| Target lags with past-only data (shift ≥ 1) | ✅ |
| Baseline + Ridge + Random Forest + XGBoost/LightGBM | ✅ |
| MAE, RMSE, R² evaluation | ✅ |
| Real vs predicted, residuals, temporal error analysis | ✅ |
| Model selection justified by val RMSE | ✅ |
| Final model saved to `models/selected/` | ✅ |
| No leakage | ✅ |